<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/18_classification/18_2_1_2_LogReg_Titanic_by_Gender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise: logistic regression - Titanic Survival by Gender

In our introduction to logistic regression, we found that sex was the strongest predictor of survival on the titanic. In this exercise, you will dig deeper by building two separate models: one for female passengers and one for male passengers.

This approach allows us to see if other features (like age or passenger class) mattered more for one gender than the other. This is a common way to explore interaction effects in your data.

### Goals:
1. Preprocess the data consistently.
2. Split the dataset into male and female subsets.
3. Train and evaluate a logistic regression model for each group.
4. Visualize and compare the odds ratios to identify differences in survival predictors.

## Step 1: Imports and Data Loading
We'll start by loading our standard machine learning stack and the titanic dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

sns.set_theme(style="whitegrid")

url = 'https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv'
df = pd.read_csv(url)
df.columns = df.columns.str.lower().str.replace(' aboard', '', regex=False)

print("Data Loaded Successfully.")
df.head()

## Step 2: Preprocessing
Before splitting the data, we must perform our feature engineering. Some of this has been started for you below to save time.

In [ ]:
# Convert pclass to integer for clean dummy column names
df['pclass'] = df['pclass'].astype(int)

# Create binary indicators for family members
df['sibspouse'] = (df['siblings/spouses'] > 0).astype(int)
df['parentchild'] = (df['parents/children'] > 0).astype(int)

# Handle the skewed distribution of fares by applying a log transformation
df['fare_log'] = np.log1p(df['fare'])

# Handle missing age values by filling with the median
df['age'] = df['age'].fillna(df['age'].median())

# TASK: Create dummy variables for 'pclass' using pd.get_dummies().
# Use drop_first=True to avoid the dummy variable trap (pclass_1 = reference).
# Hint: After encoding, print df.columns.tolist() to see the new column names.
# Your Code Here:


# TASK: Define the feature list for modeling.
# Include: age, fare_log, sibspouse, parentchild, and the pclass dummy columns.
# Remember: exclude 'sex' since we will split by gender.
# Your Code Here:



## Step 3: Segmenting, Modeling, and Evaluation
Now, perform the following tasks for both the Female and Male groups:
1. Create separate DataFrames for each gender.
2. Define features (X) and target (y) for each subset. *Note: Do not include the 'sex' column.*
3. Split into training (75%) and testing (25%) sets with `stratify=y` and `random_state=42`.
4. Build a `ColumnTransformer` that scales continuous features (`age`, `fare_log`) with `StandardScaler` and passes the remaining binary features through as-is.
5. Build a `Pipeline` with the preprocessor and `LogisticRegression(random_state=42)`.
6. Perform 5-fold cross-validation on the training set and print the mean accuracy for each model.
7. Fit the pipeline to the full training set, then print a `classification_report` and display a `ConfusionMatrixDisplay` for each model.

In [ ]:
# --- Female Model ---
# 1. Filter: df_female = df[df['sex'] == 'female']
# 2. Define X_f (features list) and y_f ('survived')
# 3. train_test_split with stratify=y_f, random_state=42
# 4. Build ColumnTransformer: StandardScaler on numeric features, remainder='passthrough'
# 5. Build Pipeline: preprocessor + LogisticRegression(random_state=42)
# 6. Run 5-fold cross-validation (cross_val_score) on the training set
# 7. Fit on the full training set, predict on test, print classification_report, display ConfusionMatrixDisplay

# Your Code Here:


# --- Male Model ---
# Repeat the same steps for male passengers

# Your Code Here:



## Step 4: Comparison of Predictors
Extract the coefficients from both models, convert them to odds ratios using `np.exp()`, and display them side-by-side in a single DataFrame.

**Hint:** Use `pipeline.named_steps['preprocessor'].get_feature_names_out()` to retrieve feature names in the correct order, then strip the `'num__'` and `'remainder__'` prefixes with `name.split('__')[-1]`.

**Hint:** Sort the final comparison DataFrame by odds ratio (e.g., female odds ratio descending) so the strongest predictors appear at the top.

In [ ]:
# Your Analysis Code Here:
# 1. Extract feature names via get_feature_names_out() from each pipeline
# 2. Strip prefixes from feature names
# 3. Get coefficients via .named_steps['classifier'].coef_[0]
# 4. Compute odds ratios with np.exp()
# 5. Create two DataFrames and merge on 'feature'
# 6. Sort by odds ratio (e.g., females descending) so the strongest predictors stand out



## Step 5: Visualizing the Interaction
Create a grouped bar chart comparing the odds ratios of the Male and Female models. A well-labeled plot with a reference line at OR=1 makes the comparison much easier to read.

**Hint:** Use `pd.melt()` to reshape from wide (one column per gender) to long format (one row per feature-gender pair) for seaborn.

**Hint:** Use `sns.barplot(data=..., x='feature', y='odds_ratio', hue='Gender')` and add a horizontal reference line at y=1 (`plt.axhline(1, color='gray', linestyle='--')`). You may also consider `plt.yscale('log')` for balanced comparison of ratios above and below 1.

In [ ]:
# Your Visualization Code Here:
# 1. Reshape the comparison DataFrame with pd.melt()
# 2. Clean up gender labels
# 3. Create grouped barplot with seaborn
# 4. Add reference line at y=1
# 5. Set log scale, rotate x labels, add title



## Discussion Questions
Based on your results, answer the following:
1. **Performance:** Which group (males or females) had a more accurate survival model? Look at the precision and recall—did the model struggle more with predicting survivors or fatalities in either group?
2. **The class effect:** Did passenger class (`pclass`) have a stronger effect on survival for men or women? How can you tell from the odds ratios?
3. **Predictive differences:** Were there any features that were predictive for one gender but not the other? (e.g., Did age matter more for one group?)
4. **Class imbalance:** Look at your confusion matrices. How did the baseline survival rate for each gender (many more women survived than men) affect the "balance" of the models?
5. **Methodology:** By splitting the data, we reduced our sample size for each model. What are the risks of training models on smaller, segmented datasets compared to one large "global" model? How might this affect the generalizability and stability of the coefficients?
6. **Stability:** Compare the cross-validation scores for the male and female models. Which model was more stable across folds? Why might this be the case given the differences in sample size?

write your discussion here:

